# Matrix Factorization (Collaborative Filtering)

In [2]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD
from surprise.model_selection import GridSearchCV, train_test_split
from surprise import accuracy

## Only keep the a few columns for the data and rerun the train/test split

In [4]:
users_movies = pd.read_csv("users_movies.csv")
r = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(users_movies[["user_id", "movie_id", "rating"]], r)

In [6]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

## Hyperparameter Tuning and 5 fold CV

In [8]:
params = {
    'biased': [False],        
    'n_epochs': [20, 30],     
    'n_factors': [50, 100], 
    'reg_all': [0.02, 0.05],
    'lr_all': [0.002, 0.005]
}

grid = GridSearchCV(SVD, params, measures=['mae', 'rmse'], cv=5, n_jobs=-1)


## Fit results to data

In [10]:
grid.fit(data)

## Analyze the best model

In [11]:
best_model = grid.best_estimator['rmse']
best_model.fit(train)
predictions = best_model.test(test)

In [14]:
accuracy.rmse(predictions)
accuracy.mae(predictions)

RMSE: 0.8570
MAE:  0.6790


0.6789553896728581

## Create train and test dataframes for use

In [16]:
train_df = pd.DataFrame(train.build_testset(), columns=["user_id", "movie_id", "rating"])
test_df = pd.DataFrame(test, columns=["user_id", "movie_id", "rating"])

In [18]:
watched = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()

t_p = (test_df[test_df["rating"] >= 4].groupby("user_id")["movie_id"].apply(set).to_dict())

all_m = set(users_movies["movie_id"].unique())

In [20]:
def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)
    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


## Test recommender on top 10 and top 100 recommendations for each user in the test set

In [23]:
total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0

for user, movies in t_p.items():
    s = watched.get(user, set())
    not_watched = all_m - s

    scores = []
    for i in not_watched:
        p = best_model.predict(user, i)
        scores.append((i, p.est))

    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)

    top_10 = []
    for movie_id, score in sorted_scores[:10]:
        top_10.append(movie_id)

    movie_set = set(movies)
    hits = len(set(top_10).intersection(movie_set))

    precision = hits / 10
    recall = hits / len(movie_set) if len(movie_set) > 0 else 0
    ndcg = ndcg_at_k(top_10, movie_set, k=10)

    total_hits += hits
    total_relevant += len(movie_set)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    num_users += 1

avg_precision = precision_sum / num_users if num_users > 0 else 0
avg_recall = recall_sum / num_users if num_users > 0 else 0
avg_ndcg = ndcg_sum / num_users if num_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", num_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")
print(f"NDCG@10: {avg_ndcg:.4f}")
print(f"Micro Recall@10: {micro_recall:.4f}")

Users evaluated: 5987
Hits: 4522
Possible hits: 115143
Precision@10: 0.0755
Recall@10: 0.0424
NDCG@10: 0.0811
Micro Recall@10: 0.0393


In [30]:
total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0

for user, movies in t_p.items():
    s = watched.get(user, set())
    not_watched = all_m - s

    scores = []
    for i in not_watched:
        p = best_model.predict(user, i)
        scores.append((i, p.est))

    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)

    top_100 = []
    for movie_id, score in sorted_scores[:100]:
        top_100.append(movie_id)

    movie_set = set(movies)
    hits = len(set(top_100).intersection(movie_set))

    precision = hits / 100
    recall = hits / len(movie_set) if len(movie_set) > 0 else 0
    ndcg = ndcg_at_k(top_100, movie_set, k=100)

    total_hits += hits
    total_relevant += len(movie_set)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    num_users += 1

avg_precision = precision_sum / num_users if num_users > 0 else 0
avg_recall = recall_sum / num_users if num_users > 0 else 0
avg_ndcg = ndcg_sum / num_users if num_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", num_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@100: {avg_precision:.4f}")
print(f"Recall@100: {avg_recall:.4f}")
print(f"NDCG@100: {avg_ndcg:.4f}")
print(f"Micro Recall@100: {micro_recall:.4f}")

Users evaluated: 5987
Hits: 24161
Possible hits: 115143
Precision@100: 0.0404
Recall@100: 0.2254
NDCG@100: 0.1353
Micro Recall@100: 0.2098
